In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [4]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [6]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/5 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [3]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files= r"V3A_clean.jsonl"

)

Generating train split: 42407 examples [00:00, 85537.45 examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [ ]:
import json
from pathlib import Path

from datasets import Dataset, Features, Value, Sequence


# ============================================================
# DATASET PATH
# ============================================================

INPUT_FILE = "./dataset/v3a_dataset_clean.jsonl"

OUTPUT_FILE = r"V3A_clean.jsonl"


# ============================================================
# V3A PROFILE FIELDS
# ============================================================

PROFILE_LIST_FIELDS = [
    "interests",
    "communication_preferences",
    "personality_traits",
    "lifestyle",
    "languages",
    "values",
    "social_preferences",
    "relationship_preferences",
    "work_preferences",
    "goals",
    "stable_habits",
    "preferences",
    "important_constraints",
]

PROFILE_STRING_FIELDS = [
    "profession",
    "career_goal",
    "education",
    "learning_style",
]


# ============================================================
# TEXT HELPERS
# ============================================================

def clean_text(value):

    if value is None:
        return ""

    return str(value).strip()


def clean_list(value):

    if value is None:
        return []

    if not isinstance(value, list):
        value = [value]

    result = []

    for item in value:

        item = clean_text(item)

        if item:
            result.append(item)

    return result


def first_text(value):

    if isinstance(value, list):

        for item in value:

            item = clean_text(item)

            if item:
                return item

        return ""

    return clean_text(value)


# ============================================================
# NORMALIZE CONVERSATION UNDERSTANDING
# ============================================================

def normalize_understanding(value):

    if not isinstance(value, dict):

        return {
            "topic": "",
            "intent": "",
        }

    # -------------------------
    # TOPIC
    # -------------------------

    topic = first_text(
        value.get("topic")
    )

    if not topic:

        topic = first_text(
            value.get("main_topic")
        )

    if not topic:

        topic = first_text(
            value.get("topics")
        )

    # -------------------------
    # INTENT
    # -------------------------

    intent = first_text(
        value.get("intent")
    )

    if not intent:

        intent = first_text(
            value.get("user_intent")
        )

    return {
        "topic": topic,
        "intent": intent,
    }


# ============================================================
# NORMALIZE PROFILE
# ============================================================

def normalize_profile(profile, line_no):

    if not isinstance(profile, dict):

        raise ValueError(
            f"Line {line_no}: output.profile is not a dictionary"
        )

    clean_profile = {}

    # -------------------------
    # STRING FIELDS
    # -------------------------

    for field in PROFILE_STRING_FIELDS:

        clean_profile[field] = clean_text(
            profile.get(field, "")
        )

    # -------------------------
    # LIST FIELDS
    # -------------------------

    for field in PROFILE_LIST_FIELDS:

        clean_profile[field] = clean_list(
            profile.get(field, [])
        )

    # -------------------------
    # CONFIDENCE
    # -------------------------

    try:

        confidence = float(
            profile.get(
                "confidence",
                0.0
            )
        )

    except (TypeError, ValueError):

        confidence = 0.0

    if confidence < 0 or confidence > 1:

        raise ValueError(
            f"Line {line_no}: confidence "
            f"must be between 0 and 1"
        )

    clean_profile["confidence"] = confidence

    return clean_profile


# ============================================================
# NORMALIZE RECORD
# ============================================================

def normalize_record(row, line_no):

    if not isinstance(row, dict):

        raise ValueError(
            f"Line {line_no}: record is not a dictionary"
        )

    # ========================================================
    # INPUT
    # ========================================================

    inp = row.get("input")

    if not isinstance(inp, dict):

        raise ValueError(
            f"Line {line_no}: input is invalid"
        )

    # ========================================================
    # OUTPUT
    # ========================================================

    output = row.get("output")

    if not isinstance(output, dict):

        raise ValueError(
            f"Line {line_no}: output is invalid"
        )

    profile = normalize_profile(
        output.get("profile"),
        line_no
    )

    # ========================================================
    # FINAL CLEAN RECORD
    # ========================================================

    record = {

        "task": clean_text(
            row.get(
                "task",
                "v3a_user_profile_intelligence"
            )
        ),

        "instruction": clean_text(
            row.get(
                "instruction",
                "Generate or update the user's long-term profile."
            )
        ),

        "input": {

            "conversation": clean_text(
                inp.get("conversation", "")
            ),

            "conversation_summary": clean_text(
                inp.get(
                    "conversation_summary",
                    ""
                )
            ),

            "conversation_understanding":
                normalize_understanding(
                    inp.get(
                        "conversation_understanding",
                        {}
                    )
                ),

            "user_memories": clean_list(
                inp.get(
                    "user_memories",
                    []
                )
            ),
        },

        "output": {

            "profile": profile

        }
    }

    return record


# ============================================================
# CLEAN DATASET
# ============================================================

def clean_dataset():

    source = Path(INPUT_FILE)
    destination = Path(OUTPUT_FILE)

    print("=" * 75)
    print("V3A DATASET CLEANING")
    print("=" * 75)

    print()
    print("INPUT:")
    print(source)

    print()
    print("OUTPUT:")
    print(destination)

    if not source.exists():

        raise FileNotFoundError(
            f"\nDataset not found:\n{source}"
        )

    total = 0
    fixed = 0
    skipped = 0

    with open(
        source,
        "r",
        encoding="utf-8"
    ) as infile, open(
        destination,
        "w",
        encoding="utf-8",
        newline="\n"
    ) as outfile:

        for line_no, line in enumerate(
            infile,
            start=1
        ):

            if not line.strip():
                continue

            try:

                raw = json.loads(line)

                old_understanding = (
                    raw
                    .get("input", {})
                    .get(
                        "conversation_understanding",
                        {}
                    )
                )

                if not (
                    isinstance(
                        old_understanding,
                        dict
                    )
                    and set(
                        old_understanding.keys()
                    ) == {
                        "topic",
                        "intent"
                    }
                ):

                    fixed += 1

                record = normalize_record(
                    raw,
                    line_no
                )

                outfile.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                        separators=(
                            ",",
                            ":"
                        )
                    )
                    + "\n"
                )

                total += 1

            except Exception as error:

                skipped += 1

                print(
                    f"SKIP line {line_no}: "
                    f"{error}"
                )

    print()
    print("=" * 75)
    print("CLEANING RESULT")
    print("=" * 75)

    print(
        f"Total records     : {total:,}"
    )

    print(
        f"Schema fixed      : {fixed:,}"
    )

    print(
        f"Skipped records   : {skipped:,}"
    )

    print()
    print(
        f"Clean file created:\n{destination}"
    )

    return destination


# ============================================================
# EXPLICIT V3A FEATURES
# ============================================================

def get_features():

    profile = {}

    # String fields

    for field in PROFILE_STRING_FIELDS:

        profile[field] = Value(
            "string"
        )

    # List fields

    for field in PROFILE_LIST_FIELDS:

        profile[field] = Sequence(
            Value("string")
        )

    # Confidence

    profile["confidence"] = Value(
        "float32"
    )

    return Features({

        "task": Value(
            "string"
        ),

        "instruction": Value(
            "string"
        ),

        "input": {

            "conversation": Value(
                "string"
            ),

            "conversation_summary": Value(
                "string"
            ),

            "conversation_understanding": {

                "topic": Value(
                    "string"
                ),

                "intent": Value(
                    "string"
                ),
            },

            "user_memories": Sequence(
                Value("string")
            ),
        },

        "output": {

            "profile": profile

        }
    })


# ============================================================
# GENERATOR
# ============================================================

def dataset_generator(clean_file):

    with open(
        clean_file,
        "r",
        encoding="utf-8"
    ) as file:

        for line_no, line in enumerate(
            file,
            start=1
        ):

            if not line.strip():
                continue

            try:

                record = json.loads(line)

                yield record

            except Exception as error:

                print(
                    f"ERROR reading line "
                    f"{line_no}: {error}"
                )


# ============================================================
# LOAD DATASET
# ============================================================

def load_v3a_dataset(clean_file):

    print()
    print("=" * 75)
    print("LOADING V3A DATASET")
    print("=" * 75)

    features = get_features()

    print()
    print("Using explicit V3A schema.")

    print()
    print("Loading with Dataset.from_generator()...")

    try:

        dataset = Dataset.from_generator(
            lambda: dataset_generator(
                clean_file
            ),
            features=features,
        )

        print()
        print("=" * 75)
        print("V3A DATASET LOAD SUCCESS")
        print("=" * 75)

        print()
        print(dataset)

        print()
        print(
            f"Number of records: "
            f"{len(dataset):,}"
        )

        print()
        print("Columns:")

        for column in dataset.column_names:

            print(
                f"  - {column}"
            )

        print()
        print("=" * 75)
        print("FIRST RECORD")
        print("=" * 75)

        print(
            json.dumps(
                dataset[0],
                indent=2,
                ensure_ascii=False
            )
        )

        return dataset

    except Exception as error:

        print()
        print("=" * 75)
        print("DATASET LOAD FAILED")
        print("=" * 75)

        print()
        print(
            "ERROR TYPE:"
        )

        print(
            type(error).__name__
        )

        print()
        print(
            "ERROR:"
        )

        print(error)

        raise


# ============================================================
# MAIN
# ============================================================

def main():

    # Step 1
    clean_file = clean_dataset()

    # Step 2
    dataset = load_v3a_dataset(
        clean_file
    )

    print()
    print("=" * 75)
    print("V3A PROCESS COMPLETE")
    print("=" * 75)

    print()
    print(
        f"Final dataset: "
        f"{len(dataset):,} records"
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

V3A DATASET CLEANING

INPUT:
dataset\v3a_dataset_clean.jsonl

OUTPUT:
V3A_clean.jsonl

CLEANING RESULT
Total records     : 67,504
Schema fixed      : 0
Skipped records   : 0

Clean file created:
V3A_clean.jsonl

LOADING V3A DATASET

Using explicit V3A schema.

Loading with Dataset.from_generator()...


Generating train split: 67504 examples [00:10, 6477.73 examples/s]



V3A DATASET LOAD SUCCESS

Dataset({
    features: ['task', 'instruction', 'input', 'output'],
    num_rows: 67504
})

Number of records: 67,504

Columns:
  - task
  - instruction
  - input
  - output

FIRST RECORD
{
  "task": "v3a_user_profile_intelligence",
  "instruction": "Generate or update the user's long-term profile.",
  "input": {
    "conversation": "Colleague: Are you still working on the branding project?\nUser: Yes, I am wrapping up the core assets. Long term, I really want to start my own boutique design agency.\nColleague: That sounds like a great path for a graphic designer.\nUser: Absolutely, typography and brand identity have always been my main passion.",
    "conversation_summary": "User discusses current branding work and long-term career aspirations of opening a design agency.",
    "conversation_understanding": {
      "topic": "",
      "intent": "career discussion"
    },
    "user_memories": []
  },
  "output": {
    "profile": {
      "profession": "Graphic D

In [ ]:
import json

# ==========================================================
# Prepare Model
# ==========================================================

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ==========================================================
# System Prompt
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant objective prediction model.

Your task is to predict the assistant's objective from the given conversation.

Rules:
- Predict only the assistant's objective.
- Do NOT generate a reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT infer unsupported information.
- Base your prediction only on the provided conversation.

Return ONLY valid JSON in the following format:

{
  "primary_objective": "",
  "secondary_objective": "",
  "priority": "",
  "reason": ""
}

Field Definitions:
- primary_objective: The assistant's main objective.
- secondary_objective: An optional supporting objective. Use "None" if no meaningful secondary objective exists.
- priority: One of "High", "Medium", or "Low".
- reason: A brief explanation (1–2 sentences) describing why these objectives were selected.

Return only the JSON object. Do not include any extra text or markdown.
"""

# ==========================================================
# Formatting Function
# ==========================================================

def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [ ]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [ ]:
old_dataset = load_dataset(
    "json",
    data_files= r"./v5a/V5A.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/59921
Processed 10000/59921
Processed 15000/59921
Processed 20000/59921
Processed 25000/59921
Processed 30000/59921
Processed 35000/59921
Processed 40000/59921
Processed 45000/59921
Processed 50000/59921
Processed 55000/59921

----- BASIC -----
Samples              : 59921
Average Tokens       : 446.63
Median Tokens        : 439
Minimum Tokens       : 303
Maximum Tokens       : 879

----- TOKEN DISTRIBUTION -----
>256 tokens          : 59921
>512 tokens          : 10744
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 87.0 sec
Average/sample       : 0.23 ms
Samples/sec          : 688.73

----- LONGEST SAMPLES -----
Sample  49784 : 879 tokens
Sample  24899 : 877 tokens
Sample  54440 : 850 tokens
Sample  26312 : 848 tokens
Sample  17506 : 842 tokens
Sample   9744 : 836 tokens
Sample  17119 : 833 tokens
Sample  39085 : 819 tokens
Sample  48582 : 818 tokens
Sample   4658 : 808 tokens
